# Week 10 build · Two candidates, one plan, one pipeline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjtri/learnml/blob/main/notebooks/week-10.ipynb)

**~3 hours · CPU only · run top to bottom.**

**The one rule:** every experiment is preceded by a 🔮 **predict cell**. Type what you expect and how sure you are *before* running the experiment. The notebook refuses empty predictions. A rough guess is fine; the gap between guess and result is what you are here to collect (canvas steps 7–8: *predict before running*, *log the surprise*).

**Two passes.** `SMOKE = True` (config cell below) renders a small dataset and uses 3 seeds, so the whole notebook runs in under five minutes on CPU. `SMOKE = False` renders the full dataset (1200 images) and sweeps 10 seeds in Part D; still under a minute of compute on CPU, so the three hours are for thinking, not waiting.

**The choice.** The pipeline defaults to the **cross-modal contrastive embedding** option (`OPTION = "contrastive"`: shapes → vOICe-style sound → spectrogram-like vectors). The **codec** alternative (shapes → 4×4 haptic grid, 8 levels) is built too. Both land in `week10_data.npz`, the file weeks 11–12 read.

| Part | Topic | Time |
|---|---|---|
| A | Render 20 shape classes; both candidates through canvas steps 1–5 | 30 min |
| B | The vOICe-style sonifier and its round-trip check | 35 min |
| C | The haptic-grid downsampler and its collision count | 20 min |
| D | Seeds vs noise: spread, effect size, how many seeds | 45 min |
| E | Choose, then write and save the one-page plan | 30 min |
| F | Stretch: week 12's trivial baseline | 20 min |

In [ ]:
#@title Setup: run me first
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display
torch.manual_seed(0); np.random.seed(0)
_LEDGER = {}

def predict(tag, prediction, confidence):
    """Log a prediction BEFORE the experiment. Refuses empty predictions."""
    if not str(prediction).strip():
        raise ValueError(f"[{tag}] Write a prediction first. A rough guess is fine; an empty one is not.")
    _LEDGER[tag] = {"prediction": str(prediction).strip(), "confidence": int(confidence), "actual": "", "surprise": ""}
    print(f"Logged [{tag}] at {confidence}% confidence: {prediction}")

def record(tag, what_happened, surprise="none"):
    if tag not in _LEDGER:
        raise ValueError(f"[{tag}] has no prediction yet. Run its predict cell first.")
    _LEDGER[tag].update(actual=str(what_happened).strip(), surprise=surprise)
    print(f"Recorded [{tag}] surprise={surprise}")

def reveal():
    rows = ["| # | prediction | sure | what happened | surprise |", "|---|---|---|---|---|"]
    rows += [f"| {t} | {r['prediction']} | {r['confidence']}% | {r['actual']} | {r['surprise']} |" for t, r in _LEDGER.items()]
    print("\n".join(rows)); display(Markdown("\n".join(rows)))

print("torch", torch.__version__, "| ready")

---
## Part A · Data, and two candidates through canvas steps 1–5 (30 min)

**Words used in this part.** *shape class*: one of 20 drawn shapes (circle, ring, cross, …) rendered at 16×16. *jitter*: random shifts of position and size so no two renders are identical. *trivial baseline*: always predict the most common class. *lever*: the one thing an idea changes (data, architecture, objective, optimization, inference).

In [ ]:
#@title Config: SMOKE = True is a five-minute CPU rehearsal; False is the full run (still CPU)
SMOKE = True  #@param {type:"boolean"}
OPTION = "contrastive"  #@param ["contrastive", "codec"]
import json, math, time, torch, torch.nn as nn, torch.nn.functional as F
cfg = dict(n_per_class=12 if SMOKE else 60, seeds=[0, 1, 2] if SMOKE else list(range(10)),
           probe_steps=200 if SMOKE else 500, sim_experiments=500 if SMOKE else 2000, seed=0)
rng = np.random.default_rng(cfg["seed"]); torch.manual_seed(cfg["seed"])
print("SMOKE:", SMOKE, "| option:", OPTION, "| config:", cfg)

In [ ]:
yy, xx = np.mgrid[0:16, 0:16].astype(float)
def _m(dx, dy, r):                                    # helpers on centred coordinates
    return dict(ax=np.abs(dx), ay=np.abs(dy), d2=dx**2 + dy**2, r=r, dx=dx, dy=dy)
SHAPES = {
 "circle":     lambda m: m["d2"] <= m["r"]**2,
 "ring":       lambda m: (m["d2"] <= m["r"]**2) & (m["d2"] >= (0.6 * m["r"])**2),
 "square":     lambda m: np.maximum(m["ax"], m["ay"]) <= m["r"],
 "hollow sq":  lambda m: (np.maximum(m["ax"], m["ay"]) <= m["r"]) & (np.maximum(m["ax"], m["ay"]) > 0.6 * m["r"]),
 "diamond":    lambda m: m["ax"] + m["ay"] <= m["r"],
 "tri up":     lambda m: (m["ay"] <= m["r"]) & (m["ax"] <= (m["dy"] + m["r"]) * 0.6),
 "tri down":   lambda m: (m["ay"] <= m["r"]) & (m["ax"] <= (m["r"] - m["dy"]) * 0.6),
 "plus":       lambda m: ((m["ax"] <= 1.2) & (m["ay"] <= m["r"])) | ((m["ay"] <= 1.2) & (m["ax"] <= m["r"])),
 "X":          lambda m: (np.abs(m["ax"] - m["ay"]) <= 1.2) & (np.maximum(m["ax"], m["ay"]) <= m["r"]),
 "h bar":      lambda m: (m["ay"] <= 1.2) & (m["ax"] <= m["r"]),
}

In [ ]:
SHAPES.update({
 "v bar":      lambda m: (m["ax"] <= 1.2) & (m["ay"] <= m["r"]),
 "rising":     lambda m: (np.abs(m["dx"] + m["dy"]) <= 1.4) & (np.maximum(m["ax"], m["ay"]) <= m["r"]),
 "falling":    lambda m: (np.abs(m["dx"] - m["dy"]) <= 1.4) & (np.maximum(m["ax"], m["ay"]) <= m["r"]),
 "L":          lambda m: ((np.abs(m["dx"] + m["r"]) <= 1.2) & (m["ay"] <= m["r"])) | ((np.abs(m["dy"] - m["r"]) <= 1.2) & (m["ax"] <= m["r"])),
 "T":          lambda m: ((np.abs(m["dy"] + m["r"]) <= 1.2) & (m["ax"] <= m["r"])) | ((m["ax"] <= 1.2) & (m["ay"] <= m["r"])),
 "U":          lambda m: ((np.abs(m["ax"] - m["r"]) <= 1.2) & (m["ay"] <= m["r"])) | ((np.abs(m["dy"] - m["r"]) <= 1.2) & (m["ax"] <= m["r"])),
 "dot":        lambda m: m["d2"] <= (0.45 * m["r"])**2,
 "half disc":  lambda m: (m["d2"] <= m["r"]**2) & (m["dy"] <= 0),
 "checker":    lambda m: (np.maximum(m["ax"], m["ay"]) <= m["r"]) & ((m["dx"] > 0) ^ (m["dy"] > 0)),
 "corners":    lambda m: (np.maximum(m["ax"], m["ay"]) <= m["r"]) & (m["ax"] > 0.5 * m["r"]) & (m["ay"] > 0.5 * m["r"]),
})
CLASSES = list(SHAPES)
print(len(CLASSES), "classes:", ", ".join(CLASSES))

In [ ]:
def render(name, rng, jitter=True):
    """One 16x16 image in [0, 1]: shape = 1, background = 0, plus faint sensor noise."""
    cx = cy = 7.5; r = 5.0
    if jitter:
        cx, cy = 7.5 + rng.uniform(-2, 2), 7.5 + rng.uniform(-2, 2); r = rng.uniform(4.0, 6.0)
    img = SHAPES[name](_m(xx - cx, yy - cy, r)).astype(float)
    return np.clip(img + 0.03 * rng.standard_normal((16, 16)), 0, 1)

def make_dataset(n_per_class, seed):
    rng = np.random.default_rng(seed)
    X = np.stack([render(c, rng) for c in CLASSES for _ in range(n_per_class)]).astype(np.float32)
    y = np.repeat(np.arange(len(CLASSES)), n_per_class)
    perm = rng.permutation(len(y))
    return X[perm], y[perm]

In [ ]:
#@title 🔮 Predict A1: N = 20 classes x n_per_class images, balanced. What does the trivial baseline score (in %)? And how many of the N images are exact duplicates of another (0, a few, many)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A1", prediction, confidence)

In [ ]:
images, labels = make_dataset(cfg["n_per_class"], cfg["seed"])
N = len(labels)
majority = np.bincount(labels).max() / N * 100
dups = N - len({(images[i] > 0.5).tobytes() for i in range(N)})     # same pixels after thresholding
print(f"{N} images of shape {images.shape[1:]}, trivial (majority-class) baseline {majority:.1f}%, duplicates: {dups}")
fig, ax = plt.subplots(2, 10, figsize=(10, 2.4))
for k, a in enumerate(ax.flat):
    a.imshow(images[labels == k][0], cmap="gray", vmin=0, vmax=1); a.set_title(CLASSES[k], fontsize=7); a.axis("off")
plt.show()

In [ ]:
#@title ✍️ A1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("A1", what_happened, surprise)

Surprise? With 20 balanced classes the trivial rung sits at 5 %: any probe below about 10 % in Part D means a broken pipeline, not a hard task. Jitter keeps duplicates near zero, which matters later: a held-out set of exact copies would flatter every method equally.

**Both candidates through canvas steps 1–5.** Fill the form for each. Defaults are a worked example; edit them until they are *yours*. The function refuses a prediction without a number.

In [ ]:
import re
PLANS = {}
def steps(name, intuition, lever, moves, prior_art_query, prediction):
    """Canvas steps 1-5 for one candidate. Step 5 must contain a number (the X)."""
    assert lever in {"data", "architecture", "objective", "optimization", "inference"}, "step 2: pick exactly one lever"
    assert re.search(r"\d", prediction), "step 5: the prediction needs a number (X)"
    PLANS[name] = dict(intuition=intuition, lever=lever, moves=moves, prior_art=prior_art_query, prediction=prediction)
    print(f"[{name}]"); [print(f"  step {i+1} {k:<11s} {v}") for i, (k, v) in enumerate(PLANS[name].items())]

In [ ]:
#@title Candidate 1 · cross-modal contrastive embedding (bridge-crossmodal)
intuition = "A sonified shape should be confusable with the shapes it sounds like, not the ones it looks like."  #@param {type:"string"}
lever = "objective"  #@param ["data", "architecture", "objective", "optimization", "inference"]
moves = "move-free-supervision (image-sound pairs come for free from the sonifier)"  #@param {type:"string"}
prior_art_query = "contrastive audio-visual embedding sensory substitution vOICe"  #@param {type:"string"}
prediction = "On 20 shapes, embedding distance predicts listener confusions better than pixel distance by >= 0.2 ordering agreement."  #@param {type:"string"}
steps("contrastive", intuition, lever, moves, prior_art_query, prediction)

In [ ]:
#@title Candidate 2 · learned image-to-touch codec (bridge-codec)
intuition = "A code shaped like the real 4x4 display, learned end to end, keeps more shape identity than a hand-designed downsampler."  #@param {type:"string"}
lever = "architecture"  #@param ["data", "architecture", "objective", "optimization", "inference"]
moves = "move-inductive-bias (the bottleneck is the display)"  #@param {type:"string"}
prior_art_query = "learned haptic encoding bottleneck vector-quantized autoencoder tactile display"  #@param {type:"string"}
prediction = "Shape accuracy from the learned 4x4 x 8-level code beats the hand-designed downsampler by >= 5 points at equal bandwidth."  #@param {type:"string"}
steps("codec", intuition, lever, moves, prior_art_query, prediction)

---
## Part B · The vOICe-style sonifier (35 min)

**Words used in this part.** *sonifier*: the fixed matrix from week 1: each image row owns a sine wave (top = high pitch), each column becomes one snippet in time, brightness sets loudness. *spectrogram-like vector*: the energy at each of the 16 pitches in each of the 16 snippets, a 16×16 map you can compare with the image. *round trip*: image → sound → map; how much of the image survives.

In [ ]:
sr, n_rows, col_dur = 8000, 16, 0.06                      # sample rate, image rows, seconds per column
t = np.arange(int(sr * col_dur)) / sr                      # time axis of one snippet (480 samples)
freqs = np.logspace(np.log10(2800), np.log10(350), n_rows)  # row 0 (top) = highest pitch
W = np.sin(2 * np.pi * freqs[None, :] * t[:, None]).astype(np.float32)   # (480, 16): week 1's matrix
NFFT = 512; fft_hz = np.fft.rfftfreq(NFFT, 1 / sr)                          # 15.6 Hz per bin
edges = np.logspace(np.log10(300), np.log10(3300), n_rows + 1)              # 16 log-spaced pitch bands
BANDS = [(fft_hz >= lo) & (fft_hz < hi) for lo, hi in zip(edges[:-1], edges[1:])]
window = np.hanning(len(t)).astype(np.float32)

def sonify(img):
    """(16, 16) image -> 1-D audio: columns become snippets in time (T = 16 * 480 samples)."""
    return (W @ img).T.reshape(-1)                          # (480, 16) -> (16, 480) -> (7680,)

def sound_map(audio):
    """audio -> (16, 16) spectrogram-like map: energy per pitch band (rows, top = high) per snippet (cols)."""
    power = np.abs(np.fft.rfft(audio.reshape(16, -1) * window, n=NFFT)) ** 2   # (16 snippets, 257 bins)
    e = np.sqrt(np.stack([power[:, b].sum(1) for b in BANDS], axis=1))          # (16 snippets, 16 bands)
    return (e / (e.max() + 1e-8)).T[::-1]                   # rows = band, flipped so row 0 = highest pitch
print("W", W.shape, "| one image ->", sonify(images[0]).shape, "samples =", 16 * col_dur, "s | bands", len(BANDS))

In [ ]:
#@title 🔮 Predict B1: Correlation between an image and the sound map of its own sound: near 1.0, about 0.7, or below 0.4? And will a rising and a falling line give different maps?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B1", prediction, confidence)

In [ ]:
def corr(a, b): return float(np.corrcoef(a.ravel(), b.ravel())[0, 1])
rising, falling = render("rising", rng, jitter=False), render("falling", rng, jitter=False)
picks = [images[labels == CLASSES.index("ring")][0], rising, falling]
fig, ax = plt.subplots(2, 3, figsize=(7.5, 4.6))
for j, img in enumerate(picks):
    m = sound_map(sonify(img))
    ax[0, j].imshow(img, cmap="gray", vmin=0, vmax=1); ax[0, j].set_title(f"image {j}", fontsize=9)
    ax[1, j].imshow(m, cmap="magma"); ax[1, j].set_title(f"sound map, corr {corr(img, m):.2f}", fontsize=9)
    for a in ax[:, j]: a.axis("off")
plt.tight_layout(); plt.show()
round_trip = np.mean([corr(im, sound_map(sonify(im))) for im in images[:100]])
print(f"mean round-trip correlation over 100 images: {round_trip:.3f}")
print(f"rising vs falling: image corr {corr(rising, falling):.2f}, sound-map corr {corr(sound_map(sonify(rising)), sound_map(sonify(falling))):.2f}")
display(Audio(sonify(rising), rate=sr))

In [ ]:
#@title ✍️ B1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B1", what_happened, surprise)

Surprise? The round trip is high but not perfect: a 60 ms snippet cannot separate neighbouring pitches cleanly (a windowed FFT has about 30 Hz of blur, and the low bands are only 50 Hz apart), so energy leaks into adjacent rows and the map is a vertically smeared image. That smear *is* the sonifier's bandwidth, the thing a learned mapping would spend differently. Rising and falling lines stay distinct because time (columns) is preserved exactly.

In [ ]:
t0 = time.time()
sound_vecs = np.stack([sound_map(sonify(im)).ravel() for im in images]).astype(np.float32)   # (N, 256)
print(f"sound vectors {sound_vecs.shape} in {time.time() - t0:.1f}s | per-image round trip: "
      f"min {min(corr(im, sv.reshape(16, 16)) for im, sv in zip(images[:200], sound_vecs[:200])):.2f}")

---
## Part C · The haptic-grid downsampler (20 min)

**Words used in this part.** *actuator grid*: a 4×4 array of tactors; each cell shows one of 8 intensity levels. *downsampler*: average each 4×4 block of the image, then round to the nearest level: the hand-designed image→touch mapping. *collision*: two images of different classes that receive the identical code; at 16 cells × 8 levels the code has 48 bits, so collisions measure what the mapping throws away, not what it could carry.

In [ ]:
#@title 🔮 Predict C1: Average-pool 16x16 -> 4x4, round to 8 levels. For what share of images is the NEAREST other code (by summed level difference) from a different class: under 10%, 10-30%, over 30%? Which class pairs collide most?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C1", prediction, confidence)

In [ ]:
def haptic_code(img, cells=4, levels=8):
    """(16, 16) -> (cells, cells) integer grid in [0, levels-1]: block average, then quantise."""
    pooled = img.reshape(cells, 16 // cells, cells, 16 // cells).mean(axis=(1, 3))
    return np.clip(np.round(pooled * (levels - 1)), 0, levels - 1).astype(np.int64)

haptic = np.stack([haptic_code(im) for im in images])                     # (N, 4, 4)
flat = haptic.reshape(N, -1)
dist = np.abs(flat[:, None, :] - flat[None, :, :]).sum(-1); np.fill_diagonal(dist, 10**6)
nearest = dist.argmin(1)                                                  # each image's closest other code
wrong = labels[nearest] != labels
print(f"{len({h.tobytes() for h in haptic})} distinct codes for {N} images; nearest code is another class for {wrong.mean() * 100:.0f}%")
pairs = {}
for i in np.flatnonzero(wrong):
    a, b = sorted((CLASSES[labels[i]], CLASSES[labels[nearest[i]]])); pairs[(a, b)] = pairs.get((a, b), 0) + 1
print("most confused pairs:", sorted(pairs.items(), key=lambda kv: -kv[1])[:5])
fig, ax = plt.subplots(1, 2, figsize=(5, 2.4)); ax[0].imshow(images[0], cmap="gray"); ax[1].imshow(haptic[0], cmap="gray", vmin=0, vmax=7)
ax[0].set_title(CLASSES[labels[0]]); ax[1].set_title("4x4, 8 levels"); [a.axis("off") for a in ax]; plt.show()

In [ ]:
#@title ✍️ C1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C1", what_happened, surprise)

Surprise? The downsampler is a *lossy* code with no learning in it: thin shapes (bars, diagonals, X) and small ones (dot) smear into similar grey blocks, so their nearest codes belong to other classes. Whether a learned code at the same 4×4 × 8 bandwidth keeps them apart is exactly the codec candidate's prediction.

**Save the pipeline.** Both representations go into one file with the labels and the seed. Later weeks load this file instead of re-rendering.

In [ ]:
np.savez_compressed("week10_data.npz", images=images, sound=sound_vecs, haptic=haptic.astype(np.int8),
                    labels=labels, classes=np.array(CLASSES), seed=cfg["seed"], option=OPTION, smoke=SMOKE)
d = np.load("week10_data.npz")
print({k: (d[k].shape, str(d[k].dtype)) for k in ["images", "sound", "haptic", "labels"]}, "| option:", str(d["option"]))
print("Colab deletes this file when the session ends: download it from the Files panel.")

---
## Part D · Seeds vs noise (45 min)

**Words used in this part.** *probe*: a tiny classifier trained on one representation (pixels, sound vectors or haptic codes) to predict the shape class; its held-out accuracy is the metric. *seed*: the number that fixes the probe's initial weights and data order; a seed sweep changes nothing else. *spread*: the standard deviation of accuracy across seeds. *effect size*: gap between two representations divided by the spread; the rule of thumb needs about 16 / Δ² seeds per row.

In [ ]:
def probe_accuracy(X, y, seed, steps=None, hidden=64, lr=1e-2):
    """Train a one-hidden-layer probe on 75% of the data with this seed; return held-out accuracy in %."""
    steps = steps or cfg["probe_steps"]
    g = torch.Generator().manual_seed(seed); torch.manual_seed(seed)
    Xt, yt = torch.tensor(X.reshape(len(X), -1), dtype=torch.float32), torch.tensor(y)
    Xt = (Xt - Xt.mean(0)) / (Xt.std(0) + 1e-6)
    perm = torch.randperm(len(yt), generator=g); n_tr = int(0.75 * len(yt))
    tr, te = perm[:n_tr], perm[n_tr:]
    net = nn.Sequential(nn.Linear(Xt.shape[1], hidden), nn.ReLU(), nn.Linear(hidden, len(CLASSES)))
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    for _ in range(steps):
        idx = tr[torch.randperm(n_tr, generator=g)[:64]]           # a mini-batch drawn with this seed
        loss = F.cross_entropy(net(Xt[idx]), yt[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return float((net(Xt[te]).argmax(1) == yt[te]).float().mean()) * 100

REPS = {"pixels": images, "sound": sound_vecs, "haptic": haptic.astype(np.float32)}
print("probe ready | seeds:", cfg["seeds"], "| steps per run:", cfg["probe_steps"])

In [ ]:
#@title 🔮 Predict D1: Train the probe on SOUND vectors once per seed. How far apart will the accuracies land (spread in points: < 1, 1-3, > 3)? And roughly what accuracy?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D1", prediction, confidence)

In [ ]:
t0 = time.time()
accs = [probe_accuracy(REPS["sound"], labels, s) for s in cfg["seeds"]]
for s, a in zip(cfg["seeds"], accs): print(f"seed {s}: {a:5.1f}%")
print(f"sound: mean {np.mean(accs):.1f}  sd {np.std(accs, ddof=1):.1f}  (n={len(accs)}, {time.time() - t0:.0f}s)")

In [ ]:
#@title ✍️ D1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict D2: Now pixels vs sound vs haptic, same seeds. Which representation wins, by how many points, and is the gap larger than the spread (effect size above 1)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D2", prediction, confidence)

In [ ]:
results = {name: [probe_accuracy(X, labels, s) for s in cfg["seeds"]] for name, X in REPS.items()}
sd_pool = np.sqrt(np.mean([np.var(v, ddof=1) for v in results.values()]))   # one spread for the table
print(f"{'representation':15s} {'mean ± sd':>14s} {'SE':>5s}   seeds")
for name, v in results.items():
    print(f"{name:15s} {np.mean(v):6.1f} ± {np.std(v, ddof=1):4.1f} {np.std(v, ddof=1)/np.sqrt(len(v)):5.1f}   {cfg['seeds']}")
print(f"\npooled spread sigma = {sd_pool:.2f} points")
base = np.mean(results["pixels"])
for name in ["sound", "haptic"]:
    gap = np.mean(results[name]) - base; delta = gap / (sd_pool + 1e-9)
    need = math.ceil(16 / delta**2) if abs(delta) > 0.05 else float("inf")
    print(f"{name:7s} vs pixels: gap {gap:+5.1f}  effect size {delta:+5.2f}  seeds needed ~ {need}")
plt.boxplot(list(results.values()), tick_labels=list(results)); plt.ylabel("held-out accuracy %"); plt.show()

In [ ]:
#@title ✍️ D2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D2", what_happened, surprise)

**Reading the table.** The `± sd` column is the standard deviation across seeds, *not* divided by √n; that is the honest version. A gap smaller than the spread is not yet evidence, whichever direction it points. The "seeds needed" column is the lesson-4 rule of thumb \(16 / \Delta^2\) applied to your own numbers: if it says 40 and you have 3, your plan needs a bigger effect or more seeds.

**Now simulate.** Take the measured spread as the truth and ask: if method A were really better than B by a gain of your choosing (the form field below; 3 points by default, or try the gap you just measured), how often would n seeds per method *show* it? "Show" means the observed gap exceeds the noise band \(2\sigma\sqrt{2/n}\).

In [ ]:
#@title 🔮 Predict D3: True gain = 3 points, spread = the pooled sd you measured. With n = 3 seeds per method, in what percentage of simulated experiments does the gap clear the noise band? And at n = 16?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D3", prediction, confidence)

In [ ]:
true_gain = 3.0  #@param {type:"number"}
gain, sigma = max(float(true_gain), 1e-6), float(sd_pool)
sim_rng = np.random.default_rng(1)
delta = gain / sigma
print(f"true gain {gain:.1f}, sigma {sigma:.2f}, effect size {delta:.2f}, rule of thumb n ~ {math.ceil(16 / delta**2) if delta > 0.05 else 'inf'}")
ns, shown, sign = [1, 2, 3, 5, 8, 16, 32, 64], [], []
for n in ns:
    a = sim_rng.normal(gain, sigma, (cfg["sim_experiments"], n)).mean(1)   # A's mean over n seeds, per experiment
    b = sim_rng.normal(0.0, sigma, (cfg["sim_experiments"], n)).mean(1)
    band = 2 * sigma * math.sqrt(2 / n)
    shown.append(np.mean(a - b > band) * 100); sign.append(np.mean(a > b) * 100)
    print(f"n = {n:2d} seeds/method: A's mean beats B's {sign[-1]:5.1f}% | gap clears the noise band {shown[-1]:5.1f}%")
plt.plot(ns, shown, "o-", label="gap clears noise band"); plt.plot(ns, sign, "s--", label="A's mean > B's mean")
plt.axhline(80, ls=":", c="grey"); plt.xscale("log"); plt.xlabel("seeds per method"); plt.ylabel("% of experiments"); plt.legend(); plt.show()

In [ ]:
#@title ✍️ D3: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D3", what_happened, surprise)

Surprise? "A's mean beats B's" is easy and nearly meaningless: with one seed it happens often even for tiny gains, and a sign is not a result. Clearing the noise band is the bar, and the seed count where that curve crosses 80 % should sit near \(16/\Delta^2\). That crossing is the seed line of your plan. Re-run the cell with `true_gain` set to the sound-vs-pixels gap you measured in D2: if that gap was near zero, the curve stays flat at the noise floor, which is what "no effect" looks like.

---
## Part E · Choose, then write the one-page plan (30 min)

**Words used in this part.** *pre-registration*: the plan written before the first run, committed to the repo so the commit hash is the time-stamp. *compute budget*: rows × seeds × minutes per run, in T4-hours; keep it at a few. *stopping rule*: a fixed step count for every row.

Choose between the two candidates from Part A. The default is the **contrastive** option; if you choose the codec, set `OPTION = "codec"` in the config cell too, so the saved file records it. Every line must be filled; the number in the prediction must be a number.

In [ ]:
#@title The plan (canvas step 5 → 6)
choice = "contrastive"  #@param ["contrastive", "codec"]
task_metric = "20 rendered shapes; agreement between embedding-distance and listener-confusion orderings"  #@param {type:"string"}
baseline = "pixel distance (trivial); pixel-loss encoder with the same sweep and steps (strong, same compute)"  #@param {type:"string"}
X = 0.2  #@param {type:"number"}
ablation = "swap the contrastive loss for a pixel loss, everything else fixed: should cost >= 0.2"  #@param {type:"string"}
stopping_rule = "3000 steps for every row, no early stopping"  #@param {type:"string"}
rows = 4  #@param {type:"integer"}
seeds_per_row = 5  #@param {type:"integer"}
minutes_per_run = 5  #@param {type:"integer"}
decision_rule = "kill if the gap < 0.1; pivot to the codec if the ablation costs nothing; scale to 50 shapes if >= 0.2"  #@param {type:"string"}

In [ ]:
plan = PLANS[choice]
blanks = [k for k, v in dict(task_metric=task_metric, baseline=baseline, ablation=ablation, stopping_rule=stopping_rule,
                             decision_rule=decision_rule).items() if not str(v).strip()]
assert not blanks and X > 0, f"plan refused: blanks {blanks}, X = {X}. Every blank is a fork you will take later."
budget_h = rows * seeds_per_row * minutes_per_run / 60
delta_min = 4 / math.sqrt(seeds_per_row)
lines = [f"# Week 12 plan  (option: {choice}, pre-registered {time.strftime('%Y-%m-%d')})", "",
         f"intuition    {plan['intuition']}", f"lever        {plan['lever']}", f"moves        {plan['moves']}",
         f"task/metric  {task_metric}", f"baseline     {baseline}", f"prediction   A beats B by >= {X} on the metric",
         f"ablation     {ablation}", f"stopping     {stopping_rule}",
         f"seeds        {seeds_per_row} per row (detects effect size >= {delta_min:.2f}; measured spread {sd_pool:.2f} pts)",
         f"budget       {rows} rows x {seeds_per_row} seeds x {minutes_per_run} min = {budget_h:.1f} T4-hours",
         f"decision     {decision_rule}", f"data         week10_data.npz (seed {cfg['seed']}, smoke={SMOKE})"]
open("week10_plan.md", "w", encoding="utf-8").write("\n".join(lines) + "\n")
print("\n".join(lines))
print("\n" + ("OK: fits one free Colab session." if budget_h <= 3 else "WARNING: over 3 T4-hours; shrink minutes or seeds."),
      "Download week10_plan.md and commit it: the hash is your time-stamp.")

---
## Part F · Stretch: week 12's trivial baseline (20 min)

**Words used in this part.** *class-pair distance*: the distance between the average image (or average sound vector) of two classes; 20 classes give 190 pairs. *ordering agreement*: the correlation between the ranks of the 190 pair distances under two representations; 1 means both order the pairs identically, 0 means no relation.

The contrastive candidate predicts that a *learned* embedding orders shape pairs more like listeners do than pixels do. Before anything is learned, ask the trivial question: do pixel distances and sound distances already order the pairs the same way? If they agree at 0.9, the learned embedding has little room to beat pixels.

In [ ]:
#@title 🔮 Predict F1: Ordering agreement between pixel-distance and sound-distance over the 190 class pairs: near 0, about 0.5, or above 0.8?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("F1", prediction, confidence)

In [ ]:
def class_means(X): return np.stack([X[labels == k].reshape(int((labels == k).sum()), -1).mean(0) for k in range(len(CLASSES))])
def pair_dists(M):
    return np.array([np.linalg.norm(M[a] - M[b]) for a in range(len(M)) for b in range(a + 1, len(M))])
def ranks(v): return np.argsort(np.argsort(v)).astype(float)
def agreement(u, v): return float(np.corrcoef(ranks(u), ranks(v))[0, 1])   # correlation of orderings
dp, ds, dh = pair_dists(class_means(images)), pair_dists(class_means(sound_vecs)), pair_dists(class_means(haptic.astype(np.float32)))
print(f"190 pairs | agreement pixel vs sound {agreement(dp, ds):.2f} | pixel vs haptic {agreement(dp, dh):.2f} | sound vs haptic {agreement(ds, dh):.2f}")
names = [f"{CLASSES[a]}-{CLASSES[b]}" for a in range(20) for b in range(a + 1, 20)]
order = np.argsort(ds)[:5]
print("closest pairs by sound:", [names[i] for i in order]); print("closest pairs by pixels:", [names[i] for i in np.argsort(dp)[:5]])
plt.scatter(ranks(dp), ranks(ds), s=8); plt.xlabel("pair rank by pixel distance"); plt.ylabel("pair rank by sound distance"); plt.show()

In [ ]:
#@title ✍️ F1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("F1", what_happened, surprise)

Surprise? Whatever the number, it is the week-12 trivial baseline: a learned embedding must beat *this* agreement with listeners, not zero. Write it into the plan's baseline line if it is not there yet.

---
## Wrap up

Run the cell below, copy the table into your insight ledger next to `build-step6`, and mark the biggest surprise. Then on the laptop:

```
python track.py done week-10/build --rating <1-5> --minutes 180 --note "biggest surprise"
python gen_week.py 11
```

In [ ]:
reveal()